<a href="https://colab.research.google.com/github/tiensdattnek-ai/colab-cg-datt/blob/main/Colab_CG_DATT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg"/></a>

# 🎮 Colab-CG-DATT — Cloud Gaming VM (Pro)

**Linux XFCE + Sunshine + Moonlight Web + Cloudflare Tunnel**, tối ưu **độ trễ thấp**.

| Tối ưu | Chi tiết |
|---|---|
| Encoder | Tự dò **NVENC / VAAPI**, preset P1 ultra-low-latency, HEVC |
| Hiển thị | Modeline khớp đúng res/fps (không scale), **tắt compositing & vsync** |
| Audio | PulseAudio `tsched=0`, fragment 5ms → buffer ~10ms |
| Mạng | **BBR + fq**, busy-poll, `tcp_notsent_lowat`, tunnel chạy **QUIC** |
| Tiến trình | Sunshine/Xorg/Pulse chạy **realtime priority**, GPU khoá xung tối đa |
| Tùy chọn | `TUNNEL=tailscale` → UDP trực tiếp, độ trễ thấp nhất |

> ⚠️ Colab ToS cấm remote desktop/game streaming. Dùng thử nghiệm, tự chịu rủi ro. Repo: https://github.com/tiensdattnek-ai/colab-cg-datt


## ① Keep-alive + kiểm tra phần cứng

In [ ]:
#@title ① Keep-alive + kiểm tra GPU { display-mode: "form" }
from IPython.display import display, HTML
display(HTML('<b>Giữ audio này phát để Colab không ngắt</b><br><audio autoplay loop controls src="https://bit.ly/49tNxyn"></audio>'))

import subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print("✅ GPU:", gpu.stdout.strip())
else:
    print("❌ CHƯA CÓ GPU! Runtime > Change runtime type > T4 GPU > Save, rồi Connect lại.")
print("libnvidia-encode:", subprocess.run("ldconfig -p | grep -c nvidia-encode", shell=True,
      capture_output=True, text=True).stdout.strip(), "thư viện")
!echo "Region: $(curl -s ipinfo.io/region) | Disk: $(df -h / | awk 'NR==2{print $4}') trống"


> 💡 **Nếu ô ① báo có T4 mà bước cài vẫn chọn `software`**: chọn `ENCODER = nvenc` ở ô cài đặt để **ép dùng GPU** (script sẽ không tự hạ xuống CPU nữa, trừ khi Sunshine thật sự không khởi động được).

In [ ]:
#@title 🧹 Reset an toàn (chạy nếu cell trước báo lỗi thư mục) { display-mode: "form" }
import os, shutil
os.chdir('/content')                      # thoat khoi thu muc sap xoa
shutil.rmtree('/content/colab-cg-datt', ignore_errors=True)
print("cwd:", os.getcwd(), "— đã dọn sạch, giờ chạy cell cài đặt bên dưới.")


In [ ]:
#@title ⚙️ Thông số máy ảo rồi bấm ▶ { display-mode: "form" }
RESOLUTION = "1920x1080" #@param ["1280x720", "1600x900", "1920x1080", "2560x1440"]
FPS        = "60"        #@param ["30", "60", "120"]
ENCODER    = "auto"      #@param ["auto", "nvenc", "vaapi", "software"]
CODEC      = "auto"      #@param ["auto", "h264", "hevc", "av1"]
TUNNEL     = "cloudflare"#@param ["cloudflare", "tailscale", "none"]
LOW_LATENCY= True        #@param {type:"boolean"}
TS_AUTHKEY = ""          #@param {type:"string"}

import os, shutil, subprocess, sys

REPO = "https://github.com/tiensdattnek-ai/colab-cg-datt"
DST  = "/content/colab-cg-datt"

# QUAN TRONG: thoat khoi thu muc truoc khi xoa, neu khong shell mat cwd
# -> "shell-init: error retrieving current directory" / "fatal: Unable to read current working directory"
os.chdir("/content")
shutil.rmtree(DST, ignore_errors=True)

r = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, DST],
                   capture_output=True, text=True)
if r.returncode or not os.path.isfile(f"{DST}/scripts/install.sh"):
    print("git clone loi:", r.stderr.strip() or "khong ro")
    print("-> thu tai zip thay the")
    subprocess.run("curl -sL %s/archive/refs/heads/main.tar.gz | tar xz -C /content && "
                   "mv /content/colab-cg-datt-main %s" % (REPO, DST), shell=True, check=True)
assert os.path.isfile(f"{DST}/scripts/install.sh"), "Khong tai duoc repo"
os.chdir(DST)
print("✅ Repo:", DST, "|", subprocess.run(["git","log","-1","--pretty=%h %s"],cwd=DST,
      capture_output=True,text=True).stdout.strip() or "zip snapshot")

W, H = RESOLUTION.split("x")
env = dict(RES_W=W, RES_H=H, FPS=FPS, ENCODER=ENCODER, CODEC=CODEC, TUNNEL=TUNNEL,
           LOWLAT="1" if LOW_LATENCY else "0", TS_AUTHKEY=TS_AUTHKEY)
os.environ.update(env)
EXPORT = " ".join(f"{k}={v!r}" for k, v in env.items())

!sudo -E env {EXPORT} bash {DST}/scripts/install.sh


In [ ]:
#@title Thông số máy ảo { display-mode: "form" }
RESOLUTION = "1920x1080" #@param ["1280x720", "1600x900", "1920x1080", "2560x1440"]
FPS        = "60"        #@param ["30", "60", "120"]
ENCODER    = "auto"      #@param ["auto", "nvenc", "vaapi", "software"]
CODEC      = "auto"      #@param ["auto", "h264", "hevc", "av1"]
TUNNEL     = "cloudflare"#@param ["cloudflare", "tailscale", "none"]
LOW_LATENCY= True        #@param {type:"boolean"}
TS_AUTHKEY = ""          #@param {type:"string"}

import os, subprocess
W,H = RESOLUTION.split("x")
os.environ.update(RES_W=W, RES_H=H, FPS=FPS, ENCODER=ENCODER, CODEC=CODEC,
                  TUNNEL=TUNNEL, LOWLAT="1" if LOW_LATENCY else "0", TS_AUTHKEY=TS_AUTHKEY)
!rm -rf /content/colab-cg-datt && git clone -q https://github.com/tiensdattnek-ai/colab-cg-datt /content/colab-cg-datt
%cd /content/colab-cg-datt
!sudo -E RES_W=$W RES_H=$H FPS=$FPS ENCODER=$ENCODER CODEC=$CODEC TUNNEL=$TUNNEL LOWLAT=${LOWLAT} TS_AUTHKEY=$TS_AUTHKEY bash scripts/install.sh


## ③ Ghép đôi Moonlight
Mở link **Choi game** ở trên → thêm host `localhost` → lấy PIN 4 số → dán vào đây.

## 🩺 Gặp lỗi? Chạy bác sĩ — tự chẩn đoán và sửa

In [ ]:
#@title 🩺 Doctor — tự chẩn đoán & sửa { display-mode: "form" }
ACTION = "doctor" #@param ["doctor", "gpu-check", "collect-logs", "verify"]
P = "/content/colab-cg-datt/scripts"
if ACTION == "doctor":        !sudo bash {P}/doctor.sh
elif ACTION == "gpu-check":   !sudo bash {P}/gpu-check.sh
elif ACTION == "collect-logs":!sudo bash {P}/collect-logs.sh
else:                         !sudo bash {P}/80-verify.sh


In [ ]:
#@title Pair { display-mode: "form" }
PIN = "" #@param {type:"string"}
!sudo bash /content/colab-cg-datt/scripts/pair.sh $PIN


## ④ Bảng điều khiển / chẩn đoán độ trễ

In [ ]:
#@title Trạng thái { display-mode: "form" }
!sudo bash /content/colab-cg-datt/scripts/status.sh


## ⑤ Cài game launcher

In [ ]:
#@title Cài Steam / Lutris / Heroic { display-mode: "form" }
LAUNCHER = "steam" #@param ["steam", "lutris", "heroic", "all"]
!sudo dpkg --add-architecture i386 && sudo apt-get update -qq
if LAUNCHER in ("steam","all"):
    !sudo apt-get install -y -qq steam-installer || sudo apt-get install -y -qq steam
if LAUNCHER in ("lutris","all"):
    !sudo apt-get install -y -qq lutris wine64
if LAUNCHER in ("heroic","all"):
    !wget -qO /tmp/h.deb $(curl -s https://api.github.com/repos/Heroic-Games-Launcher/HeroicGamesLauncher/releases/latest | grep -o 'https.*amd64.deb' | head -1) && sudo apt-get install -y -qq /tmp/h.deb
print("Xong — mở Moonlight > Desktop rồi chạy launcher trong XFCE.")


## ⑥ Backup / Restore lên Google Drive
Lưu toàn bộ game đã cài để lần sau vào chơi ngay.

In [ ]:
#@title Backup { display-mode: "form" }
ACTION = "save" #@param ["save", "load"]
from google.colab import drive; drive.mount('/content/drive')
!sudo CCG_USER=gamer bash /content/colab-cg-datt/scripts/backup.sh $ACTION /content/drive/MyDrive/ccg-backup.tar.zst


## ⑦ Log khi lỗi

In [ ]:
!sudo tail -n 25 /var/log/ccg/sunshine.log
!sudo tail -n 10 /var/log/ccg/cf-web.log /var/log/ccg/moonweb.log /var/log/ccg/xorg.log


---
### Mẹo giảm trễ thêm
1. Trong Moonlight Web: bitrate **20–30 Mbps**, bật **HEVC**, tắt **V-Sync** và **frame pacing**.
2. Dùng mạng dây / Wi-Fi 5GHz; tránh VPN chồng tunnel.
3. Nếu giật: hạ xuống `1600x900@60`, giảm bitrate về 15 Mbps.
4. Độ trễ thấp nhất: `TUNNEL=tailscale` (UDP P2P trực tiếp, bỏ qua Cloudflare edge).
